# Linear Regression

## Introduction {#sec-05-top}

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import folium
import geopandas
from itables import show

import statsmodels.api as sm
from statsmodels.formula.api import ols

from scipy import stats

from ind5003 import inference

### Example: Taiwan real estate

In [ ]:
re2 = pd.read_csv("data/taiwan_dataset.csv")
print(re2.head())

In [ ]:
#| fig-align: center
#| fig-cap: Locations of Taiwan real estate
#| label: fig-taiwan-scatter
#| fig-pos: 'ht'

plt.figure(figsize=(5,5))
ax = sns.scatterplot(data=re2, x='X', y='Y', hue='price', size='price')
ax.set_title("Taiwan real estate data");

In [ ]:
m = folium.Map(location=(45.5236, -122.6750))
df1 = geopandas.read_file('data/taiwan_dataset.csv')
gdf = geopandas.GeoDataFrame(
    re2, geometry=geopandas.points_from_xy(re2.X, re2.Y), crs="EPSG:3825"
)
gdf.explore("price", #tiles="CartoDB positron", 
            tooltip="price", marker_type="circle", 
            marker_kwds = {"radius": 50, "fill": True}, 
            legend_kwds = {"caption": "Price"})

In [ ]:
#| fig-align: center
#| fig-cap: Exploring relationship between variables, Taiwan data
#| label: fig-taiwan-eda
#| fig-pos: 'ht'

plt.figure(figsize=(9,4));
ax=plt.subplot(131)
re2.plot(x='dist_MRT', y='price', kind='scatter', ax=ax, title='distance to MRT')

ax=plt.subplot(132)
re2.plot(x='house_age', y='price', kind='scatter', ax=ax, title='House age')

ax= plt.subplot(133)
z = re2.num_stores.unique()
z.sort()
tmp2 = np.array([re2.price[re2.num_stores == x].to_numpy() for x in z], dtype=object)
ax.boxplot(tmp2, tick_labels=z); ax.set_title('Num. of stores');

## Simple Linear Regression
### Formal Set-up
### Estimation
### Hypothesis Test for Model Significance
### Coefficient of Determination, $R^2$
### Example: Price vs. house age

In [ ]:
lm_house_age_1 = ols('price ~ house_age', data=re2).fit()
lm_house_age_1.summary(slim=True)

### Example: Price vs. house age estimated line

In [ ]:
#| fig-align: center
#| fig-cap: Predicted mean price and 95% CI for varying house age.
#| label: fig-taiwan-mod1-ci
#| fig-pos: 'ht'

new_df = pd.DataFrame({'house_age' : np.linspace(0, 45, 100)})
predictions_out = lm_house_age_1.get_prediction(new_df)

# plot predicted mean and CI
ax = re2.plot(x='house_age', y='price', kind='scatter', alpha=0.5 )
ax.set_title('Price vs. age');
ax.plot(new_df.house_age, predictions_out.conf_int()[:, 0].reshape(-1), 
        color='blue', linestyle='dashed');
ax.plot(new_df.house_age, predictions_out.conf_int()[:, 1].reshape(-1), 
        color='blue', linestyle='dashed');
ax.plot(new_df.house_age, predictions_out.predicted, color='blue');

## Multiple Linear Regression
### Formal Setup
### Estimation 
### Adjusted $R^2$
### Hypothesis Tests
### Example: Price vs. house age and distance to MRT

In [ ]:
lm_age_mrt_1 = ols('price ~ house_age + dist_MRT', data=re2).fit()
lm_age_mrt_1.summary(slim=True)

### Example: Broken line regression

In [ ]:
re2['x3'] = [(x - 25) if x > 25 else 0 for x in re2.house_age]

lm_age_mrt_2 = ols('price ~ house_age + x3 + dist_MRT', data=re2).fit()
lm_age_mrt_2.summary(slim=True)

In [ ]:
#| fig-align: center
#| fig-cap: Predicted mean price and 95% CI for piecewise linear regression
#| label: fig-taiwan-mod3-broken-line
#| fig-pos: 'ht'

# make predictions
new_df = pd.DataFrame({'house_age' : np.linspace(0, 45, 100)})
new_df['x3'] = [(x - 25) if x > 25 else 0 for x in new_df.house_age]
new_df2 = new_df.copy()

new_df['dist_MRT'] = 300
new_df2['dist_MRT'] = 1500

predictions_out = lm_age_mrt_2.get_prediction(new_df)
predictions_out2 = lm_age_mrt_2.get_prediction(new_df2)

# create plot.
ax = re2.plot(x='house_age', y='price', kind='scatter', alpha=0.5 )
ax.set_title('Price vs. age');
ax.plot(new_df.house_age, predictions_out.predicted_mean, 
        color='blue', linestyle='dashed', label='dist = 300m');
ax.plot(new_df2.house_age, predictions_out2.predicted_mean, 
        color='red', linestyle='dashed', label='dist = 1500m');
ax.legend();

## Including a Categorical Variable
### Example: Price vs. num stores and distance to MRT

In [ ]:
re2['num_stores_cat'] = ['low' if x <= 4 else 'high' for x in re2.num_stores]

lm_cat_1 = ols('price ~ dist_MRT + C(num_stores_cat, Treatment("low"))', re2).fit()
lm_cat_1.summary(slim=True)

### Including an Interaction Term
### Example: Interaction between num of stores and distance to MRT

In [ ]:
lm_cat_2 = ols('price ~ dist_MRT * C(num_stores_cat, Treatment("low"))', re2).fit()
lm_cat_2.summary(slim=True)

## Residual Analysis
### Standardised Residuals
### Example: Normality check for `lm_age_mrt_1`

In [ ]:
#| fig-align: center
#| fig-cap: Histogram and qq-plot for `lm_age_mrt_1`
#| label: fig-taiwan-norm-check
#| fig-pos: 'ht'

inference.check_normality(pd.Series(lm_age_mrt_1.resid_pearson))

### Scatterplots
### Example: Residual Plots for `lm_age_mrt_2`

In [ ]:
#| fig-align: center
#| fig-cap: Residual plots for `lm_age_mrt_2`
#| label: fig-taiwan-resid-checks
#| fig-pos: 'ht'

plt.figure(figsize=(12,4));
r_s = lm_age_mrt_2.resid_pearson
ax=plt.subplot(121)
ax.scatter(re2.dist_MRT, r_s, alpha=0.5)
ax.set_xlabel('dist_mrt')
ax.axhline(y=0, color='red', linestyle='--')

ax=plt.subplot(122)
ax.scatter(re2.house_age, r_s, alpha=0.5)
ax.set_xlabel('house age');
ax.axhline(y=0, color='red', linestyle='--');

### Influential Points
### Example: Example: Influential Points for `lm_age_mrt_2` 

In [ ]:
infl = lm_age_mrt_2.get_influence()
infl_df = infl.summary_frame()
print(infl_df.sort_values(by='cooks_d', ascending=False).head())

## Transformation {#sec-regression-transformation}
### Example: Log-transformation

In [ ]:
re2['ldist'] = np.log(re2.dist_MRT)

lm_age_mrt_3 = ols('price ~ house_age + x3 + num_stores + ldist', data=re2).fit()
lm_age_mrt_3.summary(slim=True)

## Summary, Further topics
## References
### Website References
## Exercises {#sec-reg-exercises}